# VHAGAR: produce the Prithvi masks (TerraTorch, Colab GPU)

This fine-tunes **Prithvi-EO-2.0-300M-BurnScars** on your fires and writes one
`<event_id>.npy` burned mask per fire into `PRITHVI_DIR`, which the head-to-head
notebook then scores against the U-Net and RBR threshold.

Three stages: (1) export chips from your cache **[exact, from the repo]**,
(2) fine-tune + predict with TerraTorch **[adapt to your TerraTorch version]**,
(3) stitch per-chip predictions back to per-fire masks **[exact, from the repo]**.

> **Honesty note.** TerraTorch's config schema and the model-card checkpoint names
> change between releases. Stages 1 and 3 use VHAGAR's own functions and are pinned;
> stage 2's YAML is a **template** you must reconcile with the *current* official
> Prithvi-EO burn-scars config. The one thing you must not change is the 6-band order
> `(blue, green, red, nir_narrow, swir_1, swir_2)` and `-1 = nodata` in the labels.


## 0. Runtime + installs
Set *Runtime -> GPU*. TerraTorch pulls torch/lightning; rasterio is needed for the chip writes.


In [ ]:
# TerraTorch is version-sensitive on Colab. Install it + rasterio, then RESTART
# the runtime so numpy/scipy load cleanly against the resolved versions.
# Colab auto-restarts after the kill; then run the NEXT cell.
%pip install -q terratorch rasterio
print('installed - restarting runtime...')
import os; os.kill(os.getpid(), 9)


In [ ]:
# ---- run this AFTER the runtime restarts ----
REPO = 'https://github.com/Ibekwemmanuel7/VHAGAR.git'   # <- your repo
REPO_DIR = '/content/VHAGAR'
import os
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO $REPO_DIR || echo 'clone failed (private? use a token)'
import sys; sys.path.insert(0, f'{REPO_DIR}/src')
import terratorch, torch
print('terratorch', terratorch.__version__, '| GPU', torch.cuda.is_available())


## 1. Config + mount Drive


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CACHE_DIR   = '/content/drive/MyDrive/vhagar/t2_cache'       # <- your .npz cache
PATTERN     = 'mtbs_*_w15bg.npz'
CHIPS_DIR   = '/content/chips'                              # exported chips (scratch)
PRED_DIR    = '/content/preds'                              # terratorch predictions (scratch)
PRITHVI_DIR = '/content/drive/MyDrive/vhagar/prithvi_masks' # <- masks land here
CHIP, EPOCHS, VAL_FRAC, TEST_FRAC, SEED = 224, 40, 0.15, 0.15, 0
os.makedirs(PRITHVI_DIR, exist_ok=True)


## 2. Export chips  [exact]
Whole-fire grouped split; writes `CHIPS_DIR/data/*_merged.tif` + `*.mask.tif`,
`CHIPS_DIR/splits/{train,val,test}.txt`, and the `_chips.json` manifest for stitching.


In [ ]:
import glob, numpy as np
from vhagar.datasets.burned_area import T2Sample
from vhagar.eval.t2_prithvi import export_prithvi_chips
samples = {}
for p in sorted(glob.glob(f'{CACHE_DIR}/{PATTERN}')):
    s = T2Sample.load(p)
    if s.is_usable:
        samples[s.event_id] = s
print(len(samples), 'usable fires')
counts = export_prithvi_chips(samples, CHIPS_DIR, chip=CHIP, val_frac=VAL_FRAC,
                              test_frac=TEST_FRAC, seed=SEED, burn_balance=True)
print('chips per split:', counts)


## 3. TerraTorch fine-tune config  [ADAPT to your TerraTorch version]
Base this on the **official Prithvi-EO-2.0-300M-BurnScars** config (its Hugging Face
model card ships one). Keep the model/backbone block from the card; only the
`data` block below must point at the chips we just exported. Band order and
`num_classes: 2` (unburned/burned) are fixed by our export.


In [ ]:
config = f'''
# ADAPT: merge with the official Prithvi-EO-2.0-300M-BurnScars model-card config.
# Only the data module is VHAGAR-specific and shown in full here.
trainer:
  accelerator: gpu
  devices: 1
  max_epochs: {EPOCHS}
  default_root_dir: /content/tt_runs
data:
  class_path: terratorch.datamodules.GenericNonGeoSegmentationDataModule
  init_args:
    batch_size: 8
    num_workers: 2
    num_classes: 2
    dataset_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
    output_bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
    rgb_indices: [2, 1, 0]
    train_data_root: {CHIPS_DIR}/data
    val_data_root: {CHIPS_DIR}/data
    test_data_root: {CHIPS_DIR}/data
    predict_data_root: {CHIPS_DIR}/data
    train_split: {CHIPS_DIR}/splits/train.txt
    val_split: {CHIPS_DIR}/splits/val.txt
    test_split: {CHIPS_DIR}/splits/test.txt
    img_grep: '*_merged.tif'
    label_grep: '*.mask.tif'
    no_label_replace: -1
    means: [0.1,0.1,0.1,0.2,0.15,0.1]   # ADAPT: use the model card's stats
    stds:  [0.05,0.05,0.05,0.07,0.06,0.05]
# model: <-- paste the EncoderDecoderFactory / prithvi_eo_v2_300 block from the card,
#            set num_classes: 2 and in_channels to the 6 bands above.
'''
open('/content/burn_config.yaml','w').write(config)
print('wrote /content/burn_config.yaml  (merge the model block before fitting)')


## 4. Fit + predict  [TerraTorch CLI]
You may need `huggingface-cli login` for the pretrained backbone. Fine-tune, then
predict on the **test** chips. If you just want a fast zero-shot baseline, skip `fit`
and predict straight from the model-card checkpoint.


In [ ]:
# !huggingface-cli login   # if the backbone download needs auth
!terratorch fit --config /content/burn_config.yaml
# point --ckpt_path at the best checkpoint terratorch wrote under /content/tt_runs
!terratorch predict --config /content/burn_config.yaml \
    --ckpt_path /content/tt_runs/**/checkpoints/*.ckpt \
    --predict_output_dir $PRED_DIR


## 5. Stitch per-chip predictions -> per-fire masks  [exact]
Map each predicted raster back to its chip stem, stitch with the manifest, and save
one `.npy` per fire. **ADAPT** the two marked lines to your predict output's filenames
(the stem must match the exported `{fire}_{i}`).


In [ ]:
import glob, json, numpy as np, rasterio
from vhagar.eval.t2_prithvi import stitch_chip_predictions
manifest = json.load(open(f'{CHIPS_DIR}/_chips.json'))
pred_by_stem = {}
for f in glob.glob(f'{PRED_DIR}/*.tif'):
    stem = os.path.basename(f)
    for suf in ('_pred.tif', '.tif', '_merged.tif'):   # ADAPT: strip your writer's suffix
        stem = stem.replace(suf, '')
    with rasterio.open(f) as ds:
        arr = ds.read(1)                                # ADAPT: band with the class/argmax
    pred_by_stem[stem] = (arr > 0).astype('uint8')
print('parsed', len(pred_by_stem), 'chip predictions;',
      len(set(pred_by_stem) & set(manifest)), 'match the manifest')
masks = stitch_chip_predictions(pred_by_stem, manifest)
for eid, m in masks.items():
    safe = eid.replace(':','_').replace('/','_')
    np.save(f'{PRITHVI_DIR}/{safe}.npy', m)
print('saved', len(masks), 'per-fire masks to', PRITHVI_DIR)


## Done
Open the head-to-head notebook, set `PRITHVI_DIR` to the folder above, and re-run its
cell 6. The Prithvi leg now appears with its paired bootstrap CI against U-Net and RBR.

Note: the head-to-head keys masks by `event_id`; if your ids contain `:` or `/`, keep
the same `safe` substitution on both ends (it already matches the export).
